# Volatility Targeting: Inverse-Vol Weighted Portfolio

An **inverse-volatility weighting** strategy on QQQ/BIL/GLD using `Weigh.BasedOnHV`.
Each asset's weight is scaled so the portfolio's annualised volatility stays near a target.

When markets are calm, the strategy levers up (sum of weights > 1).
When markets are choppy, it scales down (sum < 1 = cash buffer).

This notebook demonstrates:
- `Weigh.BasedOnHV` for volatility-targeted allocation
- `Signal.Quarterly()` for quarterly rebalancing
- Parameter sensitivity analysis (different `target_hv` values)
- Weight evolution visualisation

**Offline mode**: Uses local CSV files — no network required.

In [ ]:
from pathlib import Path

import pandas as pd

import tiportfolio as ti

_DATA_DIR = Path("../../tests/data")

CSV_DATA: dict[str, str] = {
    "QQQ": str(_DATA_DIR / "qqq_2018_2024_yf.csv"),
    "BIL": str(_DATA_DIR / "bil_2018_2024_yf.csv"),
    "GLD": str(_DATA_DIR / "gld_2018_2024_yf.csv"),
}

TICKERS = ["QQQ", "BIL", "GLD"]
INITIAL_RATIO = {"QQQ": 0.7, "BIL": 0.2, "GLD": 0.1}
START = "2019-01-01"
END = "2024-12-31"

## 1. Load Data

In [ ]:
data = ti.fetch_data(TICKERS, start=START, end=END, csv=CSV_DATA)

for ticker, df in data.items():
    print(f"{ticker}: {df.shape[0]} rows, {df.index[0].date()} \u2192 {df.index[-1].date()}")

## 2. Volatility-Targeting Strategy (target = 15%)

`Weigh.BasedOnHV` scales an initial ratio up or down so the portfolio's annualised
volatility stays near `target_hv`. It uses a diagonal covariance approximation over
the lookback window.

- When realised vol < target \u2192 weights scale **up** (leverage)
- When realised vol > target \u2192 weights scale **down** (cash buffer)

In [ ]:
vol_target = ti.Portfolio(
    "vol_target_15pct",
    [
        ti.Signal.Quarterly(),
        ti.Select.All(),
        ti.Weigh.BasedOnHV(
            initial_ratio=INITIAL_RATIO,
            target_hv=0.15,
            lookback=pd.DateOffset(months=1),
        ),
        ti.Action.Rebalance(),
    ],
    TICKERS,
)

result = ti.run(ti.Backtest(vol_target, data))

## 3. Results

In [ ]:
result.summary()

In [ ]:
result[0].plot_interactive()

In [ ]:
result[0].plot_security_weights()

In [ ]:
result[0].trades.sample(5)

## 4. Baseline Comparisons

Compare against:
1. **Fixed 70/20/10** \u2014 same initial allocation but no vol adjustment, monthly rebalance
2. **QQQ Only** \u2014 100% QQQ buy-and-hold

In [ ]:
fixed_ratio = ti.Portfolio(
    "fixed_70_20_10",
    [ti.Signal.Monthly(), ti.Select.All(), ti.Weigh.Ratio(weights=INITIAL_RATIO), ti.Action.Rebalance()],
    TICKERS,
)

qqq_only = ti.Portfolio(
    "qqq_only",
    [ti.Signal.Once(), ti.Select.All(), ti.Weigh.Equally(), ti.Action.Rebalance()],
    ["QQQ"],
)

comparison = ti.run(
    ti.Backtest(vol_target, data),
    ti.Backtest(fixed_ratio, data),
    ti.Backtest(qqq_only, data),
)

In [ ]:
comparison.summary()

In [ ]:
comparison.plot_interactive()

## 5. Target Volatility Sensitivity

Re-run with a lower target (10%) to see how a more conservative vol target affects returns and drawdowns.

In [ ]:
vol_target_10 = ti.Portfolio(
    "vol_target_10pct",
    [
        ti.Signal.Quarterly(),
        ti.Select.All(),
        ti.Weigh.BasedOnHV(
            initial_ratio=INITIAL_RATIO,
            target_hv=0.10,
            lookback=pd.DateOffset(months=1),
        ),
        ti.Action.Rebalance(),
    ],
    TICKERS,
)

sensitivity = ti.run(
    ti.Backtest(vol_target, data),
    ti.Backtest(vol_target_10, data),
)

In [ ]:
sensitivity.summary()

In [ ]:
sensitivity.plot_interactive()